Skrypt tworzy system ekspertów wykorzystujący CrewAI do znalezienia wspólnej definicji sztucznej inteligencji akceptowalnej dla matematyka, filozofa i informatyka. System składa się z czterech agentów: badacza koordynującego (może delegować zadania), matematyka, filozofa i informatyka, którzy współpracują sekwencyjnie nad rozwiązaniem problemu.

# Setup

In [5]:
!uv pip install -qU crewai langchain_community

In [ ]:
from crewai import Agent, Task, Crew, Process
from google.colab import userdata
import os

Ten kod importuje niezbędne moduły i funkcje z różnych bibliotek, przygotowując środowisko do pracy z frameworkiem CrewAI. Rozbija się to na następujące elementy:

*   `from crewai import Agent, Task, Crew, Process`: Importuje klasy `Agent`, `Task`, `Crew` i `Process` z biblioteki `crewai`.
    *   `Agent`: Reprezentuje pojedynczego agenta w systemie CrewAI. Agenci wykonują zadania przy użyciu dostępnych narzędzi.
    *   `Task`: Definiuje konkretne zadanie, które ma zostać wykonane przez agenta.
    *   `Crew`:  Reprezentuje grupę współpracujących ze sobą agentów.
    *   `Process`: Zarządza przebiegiem pracy grupy agentów (crew) i ich interakcjami.

*   `from crewai.tools import tool`: Importuje dekorator `tool` z modułu `crewai.tools`. Ten dekorator służy do definiowania narzędzi, które agenci mogą wykorzystywać do wykonywania zadań.

*   `from google.colab import userdata`: Importuje moduł `userdata` z biblioteki `google.colab`. Pozwala to na dostęp do danych użytkownika przechowywanych w Google Colab, takich jak klucze API lub inne poufne informacje.

*   `import os`: Importuje moduł `os`, który zapewnia funkcje interakcji z systemem operacyjnym, takie jak odczytywanie zmiennych środowiskowych czy tworzenie katalogów.

In [ ]:
class CFG:
    model = "gpt-4o-mini"


os.environ["OPENAI_API_KEY"] = userdata.get("openaivision")

# Agenci

In [ ]:
researcher = Agent(
    role="Researcher",
    goal="""To collaborate with your collages to gather their views
        and options and get a final answer to complex questions""",
    verbose=True,
    allow_delegation=True,
    llm=CFG.model,
    backstory="""You are an experimented researcher.
        You collaborate with expert collages to get their views.
        You synthesize information in a simple and understandable way.
        """,
)


mathematician = Agent(
    role="Mathematician",
    goal="To collaborate with your collages and provide your views as a Mathematician",
    verbose=True,
    allow_delegation=False,
    llm=CFG.model,
    backstory="""You are an experimented applied Mathematician.
        You can find answers to complex questions and work with our colleagues.
        """,
)

philosopher = Agent(
    role="philosopher",
    goal="To collaborate with your collages and provide your views as a philosopher",
    verbose=True,
    allow_delegation=False,
    llm=CFG.model,
    backstory="""You are an experimented Philosopher.
        You can find answers to complex questions and work with our colleagues.
        """,
)

scientist = Agent(
    role="computer scientist",
    goal="To collaborate with your collages and provide your views as an computer scientist",
    verbose=True,
    allow_delegation=False,
    llm=CFG.model,
    backstory="""You are an experimented computer scientist.
        You can find answers to complex questions and work with our colleagues.
        """,
)

Ten kod definiuje cztery agentów w systemie CrewAI, każdy z przypisaną rolą i cechami charakterystycznymi. Każdy agent jest tworzony za pomocą klasy `Agent` i konfigurowany następującymi parametrami:

*   `role`: Określa rolę agenta (np. "Researcher", "Mathematician"). Ta rola wpływa na sposób, w jaki agent podchodzi do rozwiązywania problemów i komunikuje się z innymi agentami.
*   `goal`: Definiuje cel agenta. Jest to instrukcja dla agenta, co ma osiągnąć podczas współpracy z innymi agentami. Zauważ, że użyto f-stringu (f"""..."""), aby włączyć do celu dynamiczną treść.
*   `verbose = True`: Włącza tryb szczegółowy (verbose). Oznacza to, że agent będzie wyświetlał więcej informacji o swoich działaniach i procesie myślowym, co ułatwia debugowanie i zrozumienie jego zachowania.
*   `allow_delegation`: Określa, czy agent może delegować zadania innym agentom. `True` oznacza, że może, a `False` oznacza, że nie. Agent "Researcher" ma ustawioną wartość `True`, co pozwala mu na przekazywanie zadań do innych ekspertów. Pozostali agenci mają `False`, co oznacza, że wykonują zadania samodzielnie.
*   `llm = CFG.model`: Określa model językowy (LLM), którego agent ma używać. W tym przypadku, wszyscy agenci korzystają z modelu określonego w zmiennej `CFG.model` (która została wcześniej ustawiona na 'gpt-4o-mini').
*   `backstory`: Dostarcza agentowi tło i kontekst. Opisuje jego doświadczenie, umiejętności i sposób myślenia. Pomaga to modelowi językowemu lepiej zrozumieć rolę agenta i generować bardziej odpowiednie odpowiedzi.

Konkretni agenci:

*   **Researcher:** Agent badawczy, którego celem jest zbieranie opinii od współpracowników w celu znalezienia ostatecznej odpowiedzi na złożone pytania. Może delegować zadania innym agentom.
*   **Mathematician:** Agent matematyk, który ma dostarczać swoje opinie z perspektywy matematycznej. Nie może delegować zadań.
*   **Philosopher:** Agent filozof, którego celem jest dzielenie się przemyśleniami i opiniami z punktu widzenia filozofii. Nie może delegować zadań.
*   **Scientist (Computer Scientist):** Agent informatyk, który ma dostarczać swoje opinie z perspektywy informatyki. Nie może delegować zadań.


In [ ]:
task = Task(
    description="What is a definition of Artificial Intelligence  that a Mathematician, a Computer Scientist, and a Philosopher can all agree?",
    expected_output="""
    The answer to the question: "What is a definition of Artificial Intelligence that a Mathematician, a Computer Scientist, and an Philosopher can all agree?"
    """,
    max_inter=5,
    agent=researcher,
)

Ten kod definiuje zadanie dla agenta badawczego (`researcher`) w systemie CrewAI.

*   `task = Task(...)`: Tworzy instancję klasy `Task`, która reprezentuje pojedyncze zadanie do wykonania.
*   `description="What is a definition of Artificial Intelligence that a Mathematician, a Computer Scientist, and a Philosopher can all agree?"`:  Określa opis zadania. Jest to pytanie, na które agent ma znaleźć odpowiedź. Zadaniem jest znalezienie definicji sztucznej inteligencji, która byłaby akceptowalna dla matematyka, informatyka i filozofa.
*   `expected_output="""..."""`: Definiuje oczekiwany format odpowiedzi. Określa, że odpowiedź powinna być sformułowana jako bezpośrednia odpowiedź na zadane pytanie. Użycie potrójnych cudzysłowów pozwala na zdefiniowanie wieloliniowego ciągu znaków.
*   `max_inter=5`:  Ustawia maksymalną liczbę interakcji (iteracji) między agentami podczas próby rozwiązania zadania. Oznacza to, że agent badawczy może delegować podzadania i czekać na odpowiedzi od innych agentów maksymalnie 5 razy.
*   `agent=researcher`: Przypisuje zadanie do konkretnego agenta – w tym przypadku do agenta `researcher`. Agent badawczy będzie odpowiedzialny za koordynację pracy pozostałych agentów (matematyka, informatyka i filozofa) w celu znalezienia odpowiedzi na pytanie.

W skrócie, ten kod tworzy zadanie polegające na znalezieniu wspólnej definicji sztucznej inteligencji, które zostanie powierzone agentowi badawczemu do rozwiązania przy pomocy innych ekspertów. Agent ma maksymalnie 5 iteracji interakcji z innymi agentami w celu osiągnięcia celu.

# Test

In [ ]:
crew = Crew(
    agents=[researcher, mathematician, philosopher, scientist],
    tasks=[task],
    verbose=True,
    process=Process.sequential,
)

result = crew.kickoff()


╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 75540507-05c7-4848-9870-d991293be3a2                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

🚀 Crew: crew
└── 📋 Task: e8ca77be-289c-41ba-9707-ce011c2fcb2d
       Status: Executing Task...

🚀 Crew: crew
└── 📋 Task: e8ca77be-289c-41ba-9707-ce011c2fcb2d
       Status: Executing Task...
    └── 🤖 Agent: Researcher
            Status: In Progress

# Agent: Researcher
## Task: What is a definition of Artificial Intelligence  that a Mathematician, a Computer Scientist, and a Philosopher can all agree?


🤖 Agent: Researcher
    Status: In Progress

...



# Agent: Researcher
## Final Answer: 
Artificial Intelligence (AI) can be defined as a multidisciplinary field within computer science dedicated to creating systems that can perform tasks traditionally requiring human intelligence. These tasks include problem-solving, reasoning, learning, perception, and language understanding.

From a mathematical perspective, AI relies heavily on logical reasoning and algorithms, utilizing mathematical principles to process data and draw inferences. Logical reasoning encompasses the application of formal logic to derive conclusions from existing knowledge, whereas algorithms form the computational backbone that enables systems to analyze vast datasets and make predictions.

From a technological perspective, AI incorporates various programming techniques such as machine learning, deep learning, natural language processing, and computer vision. These technologies empower AI systems to learn from data patterns, understand human language, interpret vis

🚀 Crew: crew
└── 📋 Task: e8ca77be-289c-41ba-9707-ce011c2fcb2d
       Status: Executing Task...
    ├── 🤖 Agent: Researcher
    │       Status: In Progress
    ├── 🤖 Agent: Mathematician
    │       Status: ✅ Completed
    ├── 🤖 Agent: computer scientist
    │       Status: ✅ Completed
    └── 🤖 Agent: Researcher
            Status: ✅ Completed

🚀 Crew: crew
└── 📋 Task: e8ca77be-289c-41ba-9707-ce011c2fcb2d
       Assigned to: Researcher
       Status: ✅ Completed
    ├── 🤖 Agent: Researcher
    │       Status: In Progress
    ├── 🤖 Agent: Mathematician
    │       Status: ✅ Completed
    ├── 🤖 Agent: computer scientist
    │       Status: ✅ Completed
    └── 🤖 Agent: Researcher
            Status: ✅ Completed

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: e8ca77be-289c-41ba-9707-ce011c2fcb2d                                                                     │
│  Agent: Researcher                                                                                              │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 75540507-05c7-4848-9870-d991293be3a2                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Ten kod tworzy grupę agentów (crew) i uruchamia proces rozwiązywania zdefiniowanego zadania.

*   `crew = Crew(...)`: Tworzy instancję klasy `Crew`, która reprezentuje zespół współpracujących ze sobą agentów.
    *   `agents=[researcher, mathematician, philosopher, scientist]`: Określa listę agentów wchodzących w skład grupy. Są to agenci zdefiniowani wcześniej: badacz, matematyk, filozof i informatyk.
    *   `tasks=[task]`:  Określa listę zadań, które mają zostać wykonane przez grupę. W tym przypadku jest tylko jedno zadanie – znalezienie wspólnej definicji sztucznej inteligencji.
    *   `verbose=True`: Włącza tryb szczegółowy (verbose) dla całej grupy agentów. Oznacza to, że będą wyświetlane informacje o przebiegu pracy każdego agenta i interakcjach między nimi.
    *   `process=Process.sequential`: Określa sposób przetwarzania zadań przez grupę. `Process.sequential` oznacza, że zadania będą wykonywane sekwencyjnie – jeden po drugim. Agent badawczy (przypisany do zadania) będzie koordynował pracę pozostałych agentów w celu rozwiązania zadania krok po kroku.

*   `result = crew.kickoff()`: Uruchamia proces rozwiązywania zadań przez grupę agentów.
    *   `crew.kickoff()`: Wywołuje metodę `kickoff()` na obiekcie `crew`, co inicjuje pracę grupy i rozpoczyna wykonywanie zdefiniowanych zadań.
    *   `result = ...`: Przypisuje wynik działania grupy (rozwiązanie zadania) do zmiennej `result`. Zmienna ta będzie zawierać odpowiedź wygenerowaną przez agentów.

W skrócie, ten kod tworzy grupę ekspertów i zleca im rozwiązanie zdefiniowanego problemu (znalezienie definicji sztucznej inteligencji). Proces rozwiązywania odbywa się sekwencyjnie, a wynik jest przechowywany w zmiennej `result`.